In [1]:
import math
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

# -------------------------
# Datos (desde el documento TTAP del usuario)
# -------------------------
nodes = {"Base": (0,0), "A": (10,5), "B": (20,15), "C": (30,10), "D": (40,20)}

helicopters = {
    "H1": {"hovering": True,  "load": 500, "med": 3, "pers": 3, "speed": 150},
    "H2": {"hovering": False, "load": 500, "med": 3, "pers": 3, "speed": 150},
    "H3": {"hovering": True,  "load": 500, "med": 3, "pers": 3, "speed": 150},
    "H4": {"hovering": False, "load": 500, "med": 3, "pers": 3, "speed": 150},
}

task_rows = [
  # id, hover_req, transport, medical, personnel, t_opt, t_eff, t_ineff  (min)
  (1, True, 200, 2, 1, 20, 40, 60),
  (2, False,300, 1, 2, 15, 30, 50),
  (3, True, 100, 3, 0, 10, 25, 40),
  (4, False,200, 0, 3, 30, 50, 70),
  (5, True, 150, 1, 1, 25, 45, 60),
  (6, True, 100, 3, 0, 10, 25,120),
  (7, False,400, 0, 3, 30, 50, 90),
  (8, True, 150, 1, 1, 25, 45,100),
  (9, True, 200, 2, 1, 20, 40, 60),
  (10,False,300, 1, 2, 15, 30, 50),
  (11,True, 100, 3, 0, 10, 25, 40),
  (12,False,200, 0, 3, 30, 50, 70),
  (13,True, 150, 1, 1, 25, 45, 60),
  (14,True, 100, 3, 0, 10, 25,120),
  (15,False,400, 0, 3, 30, 50, 90),
  (16,True, 150, 1, 1, 25, 45,100),
  (17,True, 150, 1, 1, 25, 45, 60),
  (18,True, 100, 3, 0, 10, 25,120),
  (19,False,400, 0, 3, 30, 50, 90),
  (20,True, 150, 1, 1, 25, 45,100),
]
tasks = {}
for (i, hov, tr, med, pers, topt, teff, tineff) in task_rows:
    tasks[i] = dict(hover_req=hov, transport=tr, med=med, pers=pers,
                    t_opt=topt, t_eff=teff, t_ineff=tineff)

task_to_node = {}
for i in [1,2,9,10]: task_to_node[i] = "A"
for i in [3,8,7,11,12,13,14]: task_to_node[i] = "B"
for i in [5,6,15,16]: task_to_node[i] = "C"
for i in [4,17,18,19,20]: task_to_node[i] = "D"

# -------------------------
# Parámetros de tiempo
# -------------------------
time_unit = 0.5   # min por paso (discretización)
Tmax_min = 150
Tmax = int(Tmax_min / time_unit)       # pasos
service_min = 6
service = int(service_min / time_unit) # pasos
alpha = 0.1
v_norm = 1/len(tasks)

def euclid(a,b): return math.hypot(a[0]-b[0], a[1]-b[1])

def travel_time_min(n1,n2,speed=150):
    if n1 == n2: return 0.0
    d = euclid(nodes[n1], nodes[n2])  # asumiendo km
    return 60.0 * d / speed

travel_steps = {}
for n1 in nodes:
    for n2 in nodes:
        dt = travel_time_min(n1,n2,150)
        travel_steps[(n1,n2)] = int(round(dt / time_unit)) if n1 != n2 else 0

def satisfaction(t_min, t_opt, t_eff, t_ineff, alpha=0.1):
    if t_min <= t_opt:
        return 1.0
    if t_min >= t_ineff:
        return 0.0
    return 1.0 / (1.0 + math.exp(alpha*(t_min - t_eff)))

# -------------------------
# Construcción de arcos (wait / travel / task)
# -------------------------
arcs = []
for n in nodes:
    for t in range(Tmax):
        arcs.append({"from": (n,t), "to": (n,t+1), "kind": "wait"})

for n1 in nodes:
    for n2 in nodes:
        if n1 == n2: 
            continue
        dt = travel_steps[(n1,n2)]
        for t in range(Tmax+1):
            if t + dt <= Tmax:
                arcs.append({"from": (n1,t), "to": (n2,t+dt), "kind": "travel"})

for task_id in tasks:
    n = task_to_node[task_id]
    for t in range(Tmax+1):
        if t + service <= Tmax:
            arcs.append({"from": (n,t), "to": (n,t+service), "kind": "task", "task": task_id})

H = list(helicopters.keys())
A = arcs
n_vars = len(H) * len(A)

# -------------------------
# Índices de variables
# -------------------------
var_index = {}
k = 0
for h in H:
    for a_id in range(len(A)):
        var_index[(h,a_id)] = k
        k += 1

# -------------------------
# Objetivo: beneficio + beta*tasa_asignacion (beta opcional)
# -------------------------
beta_task = 0.001  # pon 0.0 para "beneficio puro"
c = np.zeros(n_vars)
integrality = np.ones(n_vars, dtype=int)
lb = np.zeros(n_vars)
ub = np.ones(n_vars)

for h in H:
    for a_id, a in enumerate(A):
        vid = var_index[(h,a_id)]
        if a["kind"] == "task":
            i = a["task"]
            # compatibilidad hovering
            if tasks[i]["hover_req"] and (not helicopters[h]["hovering"]):
                ub[vid] = 0.0
                continue
            # compatibilidad por capacidad (single-task)
            if (tasks[i]["transport"] > helicopters[h]["load"] or
                tasks[i]["med"] > helicopters[h]["med"] or
                tasks[i]["pers"] > helicopters[h]["pers"]):
                ub[vid] = 0.0
                continue

            t_end = a["to"][1] * time_unit  # min
            f = satisfaction(t_end, tasks[i]["t_opt"], tasks[i]["t_eff"], tasks[i]["t_ineff"], alpha)
            reward = v_norm*f + beta_task
            c[vid] = -reward

bounds = Bounds(lb, ub)

# -------------------------
# Restricciones de flujo (salidas - entradas = RHS)
# -------------------------
states = [(n,t) for n in nodes for t in range(Tmax)]  # t < Tmax
out_arcs = defaultdict(list)
in_arcs  = defaultdict(list)
for a_id, a in enumerate(A):
    if a["from"][1] < Tmax:
        out_arcs[a["from"]].append(a_id)
    if a["to"][1] < Tmax:
        in_arcs[a["to"]].append(a_id)

row_data=[]; col_data=[]; val_data=[]; rhs=[]
row = 0
for h in H:
    for st in states:
        b = 1.0 if st == ("Base",0) else 0.0
        for a_id in out_arcs.get(st, []):
            row_data.append(row); col_data.append(var_index[(h,a_id)]); val_data.append( 1.0)
        for a_id in  in_arcs.get(st, []):
            row_data.append(row); col_data.append(var_index[(h,a_id)]); val_data.append(-1.0)
        rhs.append(b)
        row += 1

Aeq = coo_matrix((val_data,(row_data,col_data)), shape=(row, n_vars)).tocsr()
flow_con = LinearConstraint(Aeq, lb=np.array(rhs), ub=np.array(rhs))

# -------------------------
# Restricción: cada tarea a lo más una vez
# -------------------------
task_arc_ids = defaultdict(list)
for a_id,a in enumerate(A):
    if a["kind"] == "task":
        task_arc_ids[a["task"]].append(a_id)

row_data=[]; col_data=[]; val_data=[]
lb2=[]; ub2=[]
row=0
for i in tasks:
    for h in H:
        for a_id in task_arc_ids[i]:
            row_data.append(row); col_data.append(var_index[(h,a_id)]); val_data.append(1.0)
    lb2.append(-np.inf); ub2.append(1.0)
    row += 1

Atask = coo_matrix((val_data,(row_data,col_data)), shape=(row, n_vars)).tocsr()
task_con = LinearConstraint(Atask, lb=np.array(lb2), ub=np.array(ub2))

# -------------------------
# Resolver MILP exacto (HiGHS)
# -------------------------
res = milp(c=c, constraints=[flow_con, task_con], integrality=integrality, bounds=bounds,
           options={"time_limit": 120})

print("Status:", res.status, res.message)
print("Objetivo total (max):", -res.fun)
print("Nodos B&B:", getattr(res, "mip_node_count", None), "Gap:", getattr(res, "mip_gap", None))

# -------------------------
# Decodificación de schedule
# -------------------------
x = res.x
chosen = {h: defaultdict(list) for h in H}
for h in H:
    for a_id,a in enumerate(A):
        if x[var_index[(h,a_id)]] > 0.5:
            chosen[h][a["from"]].append(a_id)

schedule = {h: [] for h in H}
for h in H:
    st = ("Base",0)
    while st[1] < Tmax:
        outs = chosen[h].get(st, [])
        if not outs: break
        a = A[outs[0]]
        if a["kind"] == "task":
            i = a["task"]
            schedule[h].append((i, a["from"][0], a["from"][1]*time_unit, a["to"][1]*time_unit))
        st = a["to"]

print("\nSchedule (tarea, nodo, inicio_min, fin_min):")
for h in H:
    print(h, schedule[h])


Status: 0 Optimization terminated successfully. (HiGHS Status 7: Optimal)
Objetivo total (max): 0.7194115503212638
Nodos B&B: 284 Gap: 9.670694604051716e-05

Schedule (tarea, nodo, inicio_min, fin_min):
H1 [(9, 'A', 4.5, 10.5), (1, 'A', 10.5, 16.5), (3, 'B', 22.0, 28.0), (8, 'B', 28.0, 34.0), (13, 'B', 34.0, 40.0), (14, 'B', 40.0, 46.0)]
H2 [(2, 'A', 4.5, 10.5), (19, 'D', 24.0, 30.0), (4, 'D', 30.0, 36.0)]
H3 [(6, 'C', 12.5, 18.5), (5, 'C', 18.5, 24.5), (16, 'C', 24.5, 30.5), (20, 'D', 36.0, 42.0), (17, 'D', 42.0, 48.0), (18, 'D', 48.0, 54.0), (11, 'B', 136.0, 142.0)]
H4 [(10, 'A', 4.5, 10.5), (12, 'B', 16.0, 22.0), (7, 'B', 22.0, 28.0), (15, 'C', 32.5, 38.5)]
